### Install packages and Move Data

In [ ]:
DATE = "20240717"

# Install YOLOv5
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -qr requirements.txt comet_ml  # install

%cd /content
# Download SAM weights
!mkdir -p weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P weights

# Copy Data
!mkdir -p data/{DATE}
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/Wheat-GS-Data/Wheat-GS-Data-scaled/{DATE}/*.zip data/
!for file in data/*.zip; do unzip -q "$file" -d data/; done
!rm data/*.zip

# Copy Wheat Head Detection weights
!cp /content/drive/MyDrive/Wheat-GS-Data/wheat_head_detection_model.pt -P weights

Cloning into 'yolov5'...
remote: Enumerating objects: 17274, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 17274 (delta 1), reused 0 (delta 0), pack-reused 17270 (from 3)
Receiving objects: 100% (17274/17274), 16.12 MiB | 18.47 MiB/s, done.
Resolving deltas: 100% (11862/11862), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.8/725.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.2 MB/s eta 0:

In [ ]:
!pip install pillow==10.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.1.0
    Uninstalling pillow-11.1.0:
      Successfully uninstalled pillow-11.1.0


In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-6mfng85z
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-6mfng85z
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=1bb2d554d6c319055924f20d6145e02472d3a19eb97ba7ee706761793fa5e43d
  Stored in directory: /tmp/pip-ephem-wheel-cache-fmoco8s0/wheels/15/d7/bd/05f5f23b7dcbe70cbc6783b06f12143b0cf1a5da5c7b52dcc5
Successfully built segment_anything


In [ ]:
import PIL
import torch
import gc
import os
import glob
import numpy as np
from PIL import Image
import pandas
import cv2
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
print(PIL.__version__)

10.3.0


#### SAM-visualization functions

In [ ]:
import colorsys

def id2rgb(id, max_num_obj=256):
    if not 0 <= id <= max_num_obj:
        raise ValueError("ID should be in range(0, max_num_obj)")
    # Convert the ID into a hue value
    golden_ratio = 1.6180339887
    h = ((id * golden_ratio) % 1)           # Ensure value is between 0 and 1
    s = 0.5 + (id % 2) * 0.5       # Alternate between 0.5 and 1.0
    l = 0.5
    # Use colorsys to convert HSL to RGB
    rgb = np.zeros((3, ), dtype=np.uint8)
    if id==0:   #invalid region
        return rgb
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    rgb[0], rgb[1], rgb[2] = int(r*255), int(g*255), int(b*255)
    return rgb

# for visualize color mask
def visualize_obj(objects):
    assert len(objects.shape) == 2
    rgb_mask = np.zeros((*objects.shape[-2:], 3), dtype=np.uint8)
    all_obj_ids = np.unique(objects)
    for id in all_obj_ids:
        colored_mask = id2rgb(id)
        rgb_mask[objects == id] = colored_mask
    return rgb_mask

#### List imgae folders

In [ ]:
image_folders_pattern = os.path.join('/content/data', '*', 'images')
image_folders = sorted(glob.glob(image_folders_pattern))

print(f"All image folders: {image_folders}")

All image folders: ['/content/data/plot_461/images', '/content/data/plot_462/images', '/content/data/plot_463/images', '/content/data/plot_464/images', '/content/data/plot_465/images', '/content/data/plot_466/images', '/content/data/plot_467/images']


In [ ]:
import warnings
warnings.filterwarnings("ignore")

for folder in image_folders:
    %cd {folder}
    PLOT = folder.split('/')[-2]
    !PLOT={PLOT}
    !echo $PLOT
    print("Plot: ", PLOT)
    model = torch.hub.load('/content/yolov5', 'custom', path="/content/weights/wheat_head_detection_model.pt", source='local')

    model.conf = 0.05
    print(f"Set model confidence threshold to {model.conf}")

    data_name = os.path.basename(os.path.dirname(folder))
    print(f"Start YOLO on Data folder {data_name}")
    image_files_pattern = os.path.join(folder, '*')
    image_files = glob.glob(image_files_pattern)

    yolo_vis_folder = folder.replace("/images", "/yolo_vis")
    yolo_results_folder = folder.replace("/images", "/yolo_results")
    bbox_folder = folder.replace("/images", "/bboxes")

    print(yolo_results_folder)

    if not os.path.exists(yolo_results_folder):
        os.makedirs(yolo_results_folder)
    if not os.path.exists(yolo_vis_folder):
        os.makedirs(yolo_vis_folder)
    if not os.path.exists(bbox_folder):
        os.makedirs(bbox_folder)

    for image_file in image_files:
        # predict bounding boxes with yolo
        results = model(image_file)
        results.save(labels=False, save_dir=yolo_vis_folder, exist_ok=True)
        # results = results.pandas().xyxy[0]
        # save_name = os.path.splitext(os.path.basename(image_file))[0] + ".pkl" # ".pt"
        # results.to_pickle(os.path.join(yolo_results_folder, save_name))

        # predicted_boxes = torch.from_numpy(results.iloc[:, :4].to_numpy())
        # torch.save(predicted_boxes, os.path.join(bbox_folder, os.path.splitext(os.path.basename(image_file))[0] + ".pt"))
        break
    break

    del model
    torch.cuda.empty_cache()
    gc.collect()

    sam = sam_model_registry["vit_h"](checkpoint="/content/weights/sam_vit_h_4b8939.pth").to(device=torch.device('cuda'))
    predictor = SamPredictor(sam)

    print(f"Start SAM on Data folder {data_name}")
    sam_vis_folder = folder.replace("/images", "/sam_vis")
    mask_folder = folder.replace("/images", "/masks")

    if not os.path.exists(sam_vis_folder):
        os.makedirs(sam_vis_folder)
    if not os.path.exists(mask_folder):
        os.makedirs(mask_folder)

    for i, image_file in enumerate(image_files):
        with torch.no_grad():
            image_name = os.path.basename(image_file)
            print(i, image_name)
            if os.path.exists(os.path.join(os.path.join(sam_vis_folder, image_name))):
                print("File exists")
                pass
            else:
                image = cv2.imread(image_file)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                bbox = torch.load(os.path.join(bbox_folder, os.path.splitext(image_name)[0] + ".pt"))
                predictor.set_image(image)
                transformed_boxes = predictor.transform.apply_boxes_torch(bbox, image.shape[:2]).to('cuda')
                masks, scores, logits = predictor.predict_torch(
                    boxes = transformed_boxes,
                    multimask_output=False,
                    point_coords=None,
                    point_labels=None
                )
                assert bbox.shape[0] == masks.shape[0]
                # For colored all-masks visualization
                objects = np.zeros((masks.size(2), masks.size(3)))
                # Iterate through masks and save each individualy
                for i, mask in enumerate(masks.cpu().numpy()):
                    objects[mask.squeeze()] = i + 1
                    mask = (mask.squeeze() * 255).astype(np.uint8)
                    Image.fromarray(mask).save(os.path.join(mask_folder, f"{os.path.splitext(os.path.basename(image_file))[0]}_{i:03}.png"))
                # Save colored all-masks
                color_mask = visualize_obj(objects.astype(np.uint8)) / 255.0
                color_img = image / 255.0
                non_black_pixels = np.any(color_mask > 0, axis=-1)
                overlayed_img = color_img.copy()
                alpha = 0.6
                overlayed_img[non_black_pixels, :] = (alpha * color_mask[non_black_pixels, :] + (1 - alpha) * color_img[non_black_pixels, :])
                Image.fromarray((overlayed_img * 255).astype(np.uint8)).save(os.path.join(sam_vis_folder, image_name.replace(".png", ".jpg")))

                predictor.reset_image()
                gc.collect()
                torch.cuda.empty_cache()
    %cd ..
    !pwd
    !du -ah --max-depth=1 | sort -hr
    !find . -maxdepth 1 ! -name "masks" ! -name "bboxes" ! -name "sam_vis" ! -name "yolo_vis" ! -name "." -exec rm -rf {} +
    !zip -qr {PLOT}_yolosam.zip .
    !mv {PLOT}_yolosam.zip /content/
    %cd /content
    !cp {PLOT}_yolosam.zip /content/drive/MyDrive/Wheat-GS-Data/Wheat-GS-Data-scaled/{DATE}/yolosam_conf02/

/content/data/plot_461/images
plot_461


YOLOv5 🚀 v7.0-399-g8cc44963 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)



Plot:  plot_461


Fusing layers... 
Model summary: 476 layers, 87198694 parameters, 0 gradients
Adding AutoShape... 


Set model confidence threshold to 0.05
Start YOLO on Data folder plot_461
/content/data/plot_461/yolo_results


Saved 1 image to /content/data/plot_461/yolo_vis


In [ ]:
help(results.save)

Help on method save in module models.common:

save(labels=True, save_dir='runs/detect/exp', exist_ok=False) method of models.common.Detections instance
    Saves detection results with optional labels to a specified directory.
    
    Usage: save(labels=True, save_dir='runs/detect/exp', exist_ok=False)



In [ ]:
%cd /content

In [ ]:
!PLOT='plot_461'

In [ ]:
!echo $PLOT

In [ ]:
%cd /content
!cp plot_461_yolosam.zip /content/drive/MyDrive/Wheat-GS-Data/Wheat-GS-Data-scaled/{DATE}/yolosam/

/content
cp: cannot create regular file '/content/drive/MyDrive/Wheat-GS-Data/Wheat-GS-Data-scaled/{DATE}/yolosam/': No such file or directory


In [ ]:
!ls

In [ ]:
%cd /content
!cp plot_467_yolosam.zip /content/drive/MyDrive/Wheat-GS-Data/Wheat-GS-Data-scaled/20240719/yolosam_conf02/

/content
